# What to Freeze: A Unified Benchmark for SAM Fine-Tuning Strategies in Medical Image Segmentation

**Problem:** The Segment Anything Model (SAM) has been adapted for medical imaging through several fine-tuning strategies -- MedSAM, PP-SAM, PT-SAM -- but each paper evaluates on different datasets, with different splits, metrics, and preprocessing. No fair head-to-head comparison exists under identical conditions.

**What this notebook does:** Runs all four strategies (including a zero-shot Base SAM baseline) under a standardized evaluation protocol on the ETIS-LaribPolypDB colonoscopy dataset (196 images, 160/36 train/test split, seed=42). Training hyperparameters are per-strategy (paper-recommended), but the dataset, split, preprocessing, and evaluation metrics are identical across all strategies.

**Repository:** [github.com/nethum529/what_to_freeze](https://github.com/nethum529/what_to_freeze)

## Strategy Overview

| Strategy | Frozen | Trained | Trainable Params | Epochs | LR | WD | Loss (CE:Dice) | Augmentation | BBox Shift |
|----------|--------|---------|----------------:|-------:|----:|-----:|:--------------:|:------------:|:----------:|
| **Base SAM** | Everything | Nothing (zero-shot) | 0 | N/A | N/A | N/A | N/A | None | 20px |
| **MedSAM** | prompt_encoder | image_encoder + mask_decoder | ~93.7M | 25 | 1e-4 | 0.01 | 1.0 : 1.0 | None | 20px |
| **PP-SAM** | mask_decoder | image_encoder + prompt_encoder | ~89.7M | 100 | 1e-4 | 1e-4 | 1.0 : 1.0 | None | 50px |
| **PT-SAM** | Everything | 8 learned prompt tokens | ~2,048 | 312 | 0.05 | 0.01 | 0.2 : 0.8 | Heavy (albumentations) | 20px |

**Key design choice:** Training hyperparameters are *part of each method* -- a strategy with 2K parameters and one with 93M parameters cannot share the same learning rate and epoch count. What is standardized is the *evaluation protocol*: same dataset, same split (seed=42), same metrics (Dice, IoU, HD95), same preprocessing (1024x1024 normalized).

## Environment Setup

In [ ]:
%pip install -q monai>=1.3.0 albumentations>=1.3.0

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/what_to_freeze"

if not os.path.exists(REPO_DIR):
    # Skip LFS smudge -- LFS files are stale trained checkpoints we'll regenerate
    env = os.environ.copy()
    env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(
        ["git", "clone", "https://github.com/nethum529/what_to_freeze.git", REPO_DIR],
        env=env, check=True,
    )

# Install MedSAM (provides segment_anything package)
%pip install -q -e /kaggle/working/what_to_freeze/MedSAM-main

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
SAM_CKPT = "work_dir/SAM/sam_vit_b_01ec64.pth"
os.makedirs("work_dir/SAM", exist_ok=True)

if not os.path.exists(SAM_CKPT) and not os.path.islink(SAM_CKPT):
    # Option 1: Kaggle dataset mount (add "sam-vit-b" dataset to notebook)
    kaggle_paths = [
        "/kaggle/input/sam-vit-b/sam_vit_b_01ec64.pth",
        "/kaggle/input/sam-vit-b-01ec64/sam_vit_b_01ec64.pth",
        "/kaggle/input/sam-checkpoint/sam_vit_b_01ec64.pth",
    ]
    linked = False
    for p in kaggle_paths:
        if os.path.exists(p):
            os.symlink(p, SAM_CKPT)
            print(f"Linked SAM checkpoint from {p}")
            linked = True
            break

    # Option 2: Download from Meta AI (requires internet enabled in notebook settings)
    if not linked:
        print("Downloading SAM ViT-B checkpoint (~375 MB)...")
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", SAM_CKPT,
             "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"],
            check=True,
        )

assert os.path.exists(SAM_CKPT), f"SAM checkpoint not found at {SAM_CKPT}"
print(f"SAM checkpoint ready: {os.path.getsize(SAM_CKPT) / 1e6:.0f} MB")

In [ ]:
import torch

# Check ETIS dataset
assert os.path.exists("data/ETIS-LaribPolypDB/images"), "ETIS images not found -- check git clone"
assert os.path.exists("data/ETIS-LaribPolypDB/masks"), "ETIS masks not found -- check git clone"
n_images = len([f for f in os.listdir("data/ETIS-LaribPolypDB/images") if f.endswith(".png")])
assert n_images == 196, f"Expected 196 ETIS images, got {n_images} -- clone may be incomplete"
print(f"ETIS dataset: {n_images} images")

# Check GPU
print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")
    print("Enable GPU: Settings > Accelerator > GPU T4")

# Check SAM checkpoint size
ckpt_mb = os.path.getsize(SAM_CKPT) / 1e6
print(f"\nSAM checkpoint: {ckpt_mb:.0f} MB")
assert ckpt_mb > 300, f"SAM checkpoint seems too small ({ckpt_mb:.0f} MB), may be corrupted"
print("\nAll prerequisites verified.")

## Data Preprocessing

Convert 196 raw ETIS PNG images and masks to 1024x1024 numpy arrays, then create a reproducible 160/36 train/test split (seed=42). The split is saved to `data/npy/ETIS_split.json` for reproducibility.

In [ ]:
subprocess.run(["python", "scripts/preprocess_etis.py"], check=True)

## Benchmark Execution

The orchestrator script `run_benchmark.py` executes the full pipeline:

1. **Base SAM** -- Zero-shot evaluation (no training, ~2 minutes)
2. **PT-SAM** -- Cache encoder embeddings, then train 312 epochs (~25 minutes)
3. **MedSAM** -- Train encoder + decoder for 25 epochs (~36 minutes)
4. **PP-SAM** -- Train encoder + prompt encoder for 100 epochs (~2 hours)
5. **Evaluate** all four strategies on the 36-image test set
6. **Comparison table** with Dice, IoU, HD95, inference time, peak memory

**Estimated total time:** ~3 hours on a T4 GPU.

**Note:** The Kaggle session continues running in the background if you close the browser tab. There is no early stopping -- epoch counts are fixed per strategy for fair, reproducible benchmarking.

In [ ]:
subprocess.run(
    "python scripts/run_benchmark.py 2>&1 | tee benchmark_output.log",
    shell=True, check=True,
)

## Results Analysis

The benchmark has completed. Let's load the results and visualize performance across all four strategies.

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open("results/etis/benchmark_results.json") as f:
    results = json.load(f)

# Load training results for params/timing
train_results = {}
for strategy in ["medsam", "ppsam", "ptsam"]:
    path = f"work_dir/benchmark_etis/{strategy}/train_result.json"
    if os.path.exists(path):
        with open(path) as f:
            train_results[strategy] = json.load(f)

# Build comparison table
rows = []
strategy_order = ["basesam", "medsam", "ppsam", "ptsam"]
labels = {"basesam": "Base SAM", "medsam": "MedSAM", "ppsam": "PP-SAM", "ptsam": "PT-SAM"}

for s in strategy_order:
    if s not in results:
        continue
    summary = results[s]["summary"]
    t = train_results.get(s, {})
    params = t.get("trainable_params", 0)
    train_time = t.get("total_time_seconds", 0)

    if params >= 1e6:
        params_str = f"{params/1e6:.1f}M"
    elif params >= 1e3:
        params_str = f"{params/1e3:.1f}K"
    elif params == 0:
        params_str = "0 (zero-shot)"
    else:
        params_str = str(params)

    rows.append({
        "Strategy": labels[s],
        "Trainable Params": params_str,
        "Dice": f"{summary['dice_mean']:.4f} +/- {summary['dice_std']:.4f}",
        "IoU": f"{summary['iou_mean']:.4f} +/- {summary['iou_std']:.4f}",
        "HD95": f"{summary['hd95_mean']:.2f} +/- {summary['hd95_std']:.2f}",
        "Inference (ms)": f"{summary['inference_ms_mean']:.1f}",
        "Train Time": f"{train_time/60:.1f} min" if train_time > 0 else "N/A",
        "Peak Mem (MB)": f"{summary['peak_memory_mb']:.0f}",
    })

df = pd.DataFrame(rows)
display(df.style.set_caption("ETIS-LaribPolypDB Benchmark Results").hide(axis="index"))

## Training Curves

Training and validation loss curves for the three trainable strategies. Base SAM is excluded (zero-shot, no training).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 150, "font.size": 11})

COLORS = {"medsam": "#2196F3", "ppsam": "#FF9800", "ptsam": "#4CAF50", "basesam": "#9E9E9E"}
LABELS = {
    "medsam": "MedSAM (Enc+Dec)",
    "ppsam": "PP-SAM (Enc+Prompt)",
    "ptsam": "PT-SAM (Tokens only)",
    "basesam": "Base SAM (Zero-shot)",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for strategy in ["medsam", "ppsam", "ptsam"]:
    train_path = f"work_dir/benchmark_etis/{strategy}/train_losses.npy"
    val_path = f"work_dir/benchmark_etis/{strategy}/val_losses.npy"
    if not os.path.exists(train_path):
        continue
    train_loss = np.load(train_path)
    val_loss = np.load(val_path)
    epochs = range(1, len(train_loss) + 1)
    axes[0].plot(epochs, train_loss, color=COLORS[strategy], label=LABELS[strategy], linewidth=2)
    axes[1].plot(epochs, val_loss, color=COLORS[strategy], label=LABELS[strategy], linewidth=2)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Validation Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("ETIS-LaribPolypDB: Training Curves", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Metric Comparison

Dice, IoU, and HD95 across all four strategies. Error bars show standard deviation over the test set.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

available = [s for s in strategy_order if s in results]
bar_labels = [LABELS[s] for s in available]
bar_colors = [COLORS[s] for s in available]

for ax, metric_mean, metric_std, ylabel, title in [
    (axes[0], "dice_mean", "dice_std", "Dice Score", "Dice Coefficient"),
    (axes[1], "iou_mean", "iou_std", "IoU Score", "Intersection over Union"),
    (axes[2], "hd95_mean", "hd95_std", "HD95 (pixels)", "Hausdorff Distance 95%"),
]:
    means = [results[s]["summary"][metric_mean] for s in available]
    stds = [results[s]["summary"][metric_std] for s in available]
    bars = ax.bar(
        bar_labels, means, yerr=stds, color=bar_colors, capsize=8,
        edgecolor="black", linewidth=0.8, alpha=0.85,
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    for bar, m in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{m:.3f}", ha="center", fontsize=9, fontweight="bold",
        )

n_test = results[available[0]]["summary"]["n_images"]
fig.suptitle(f"ETIS-LaribPolypDB Test Set (N={n_test})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Per-Image Dice Distribution

Distribution of per-image Dice scores reveals variance in strategy performance. A strategy with a high mean but wide spread may be unreliable on certain image types.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

available = [s for s in strategy_order if s in results]
data = [[r["dice"] for r in results[s]["per_image"]] for s in available]
box_labels = [LABELS[s] for s in available]
box_colors = [COLORS[s] for s in available]

bp = ax.boxplot(
    data, labels=box_labels, patch_artist=True, widths=0.6,
    medianprops=dict(color="black", linewidth=2),
)
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("Dice Score")
ax.set_title("Per-Image Dice Score Distribution")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

## Parameter Efficiency

How does the number of trainable parameters relate to segmentation quality? PT-SAM trains only ~2K parameters (8 learned prompt tokens) versus ~93M for MedSAM, making it orders of magnitude more parameter-efficient.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for s in available:
    if s not in results:
        continue
    dice = results[s]["summary"]["dice_mean"]
    dice_std = results[s]["summary"]["dice_std"]
    t = train_results.get(s, {})
    params = t.get("trainable_params", 0)

    # For BaseSAM (0 params), place at x=1 on log scale with annotation
    x = params if params > 0 else 1
    ax.errorbar(
        x, dice, yerr=dice_std, fmt="o", color=COLORS[s],
        markersize=12, capsize=8, linewidth=2, label=LABELS[s],
        markeredgecolor="black", markeredgewidth=1,
    )
    if params == 0:
        ax.annotate(
            "0 params\n(zero-shot)", xy=(x, dice), xytext=(5, 30),
            textcoords="offset points", fontsize=8, ha="left",
            arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
        )

ax.set_xscale("log")
ax.set_xlabel("Trainable Parameters (log scale)")
ax.set_ylabel("Dice Score")
ax.set_title("Parameter Efficiency: Trainable Params vs. Dice")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

### Key Findings

This benchmark provides the first head-to-head comparison of four SAM adaptation strategies under identical evaluation conditions on the ETIS-LaribPolypDB colonoscopy dataset.

- **Base SAM (zero-shot)** establishes a lower bound -- SAM without any medical-domain fine-tuning. The gap between Base SAM and the fine-tuned strategies quantifies the value of adaptation.
- **MedSAM** fine-tunes the image encoder and mask decoder (~93.7M parameters), following the approach from Ma et al. (Nature Communications, 2024).
- **PP-SAM** fine-tunes the image encoder and prompt encoder (~89.7M parameters), freezing the mask decoder, as proposed by Rahman et al. (CVPRW, 2024).
- **PT-SAM** freezes the entire model and learns only 8 prompt tokens (~2K parameters), achieving high parameter efficiency via the approach from Piater et al. (CVPRW, 2025).

### Limitations

- **Single dataset:** ETIS-LaribPolypDB contains only 196 images from a single modality (colonoscopy). Results may not generalize to CT, MRI, or other imaging modalities.
- **Small test set:** With 36 test images, confidence intervals are wide. Per-image variance (visible in the box plots) is important context for interpreting mean metrics.
- **Single architecture:** All strategies use ViT-B. Results may differ with ViT-L or ViT-H backbones.

### Future Directions

- Extend to additional datasets (Kvasir-SEG, CVC-ClinicDB, CT/MRI modalities)
- Add LoRA-based adaptation strategies for comparison
- Develop novel efficiency metrics that capture the parameter-accuracy tradeoff
- Investigate ensemble approaches combining multiple freeze strategies